# Stateless design, functional programming, and JAX tracing
## Why one definition of `dx/dt = f(x, u, t, params)` is enough

This notebook explains a design choice at the heart of `minilink` and shows, with runnable code, why it is so powerful in practice.

**The idea.** In `minilink` a dynamical system is described by two *pure functions*:

$$\dot{x} = f(x, u, t; p) \qquad y = h(x, u, t; p)$$

There is no hidden state and no mutation of `self` while computing a step: `f` maps the inputs `(x, u, t, params)` to the state derivative `dx` and nothing else.

**The payoff.** Because `f` is a pure function, *automatic differentiation* can compute its derivative with respect to **any** argument — the state `x`, the input `u`, or the parameters `params` — exactly (to machine precision) and cheaply. That single capability is the engine behind linearization, control design, state/parameter estimation, identification, tuning, trajectory optimization, and model-predictive control. You write the physics **once**; the gradients come for free.

**Contents**
1. The stateless / functional contract
2. A textbook model, written once
3. What is JAX tracing?
4. ∂f/∂x and ∂f/∂u — linearization (the A and B matrices)
5. ∂f/∂params — parameter sensitivity
6. Use case: parameter identification & tuning
7. Use case: optimization & planning (differentiate through a rollout)
8. Recap

## 1. The stateless / functional contract

Every block, plant, and controller in `minilink` extends a base `System` whose contract is exactly the two equations above. The source (`minilink/core/system.py`) is explicit about the intent:

> The intended dynamical contract is **functional/stateless in intent**: overridden `f` and `h` should behave as functions of `(x, u, t, params)` only. [...] users should avoid relying on hidden mutable instance state inside dynamics or output functions.

Contrast this with a typical object-oriented simulator, where stepping the model **mutates** internal fields (`self.x = ...`, `self.t += dt`, integrator buffers, cached forces). That hidden state is convenient until you want to do anything *other* than run the simulation forward: it makes results depend on call order, blocks parallel evaluation, and — crucially — makes the model opaque to a differentiation engine.

Keeping `f` pure buys three things at once:

- **Reproducibility & composability.** `f(x, u, t, p)` always returns the same answer for the same arguments, so systems compose into diagrams cleanly and a trajectory is just `x0` plus a sequence of inputs.
- **Parallelism.** A pure function can be evaluated over a whole batch of states/inputs at once (`jax.vmap`) with no aliasing hazards — the basis of vectorized rollouts and collocation.
- **Differentiability.** A pure function can be *traced* and differentiated. This is what the rest of the notebook is about.

## 2. A textbook model, written once

We subclass `DynamicSystem` and implement `f(x, u, t, params)` in textbook style for a single actuated pendulum. The state is `x = [theta, dtheta]`, the input is the applied torque, and every physical constant lives in `params`.

The one idiom worth noting is `array_module(x)`: it returns `numpy` for ordinary arrays and `jax.numpy` for JAX arrays. So the **same** `f` runs as plain NumPy during everyday simulation, and becomes a JAX-traceable function the moment it is handed JAX values — no second implementation, no `if jax:` branches.

In [ ]:
import numpy as np

from minilink.core.system import DynamicSystem
from minilink.core.backends import array_module


class Pendulum(DynamicSystem):
    """Single actuated pendulum with every physical constant kept in params.

    State is x = [theta, dtheta]; the input is the applied torque. Writing f
    with array_module(x) keeps one definition working under NumPy and JAX.
    """

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2, y_dependencies=())
        self.name = "Pendulum"
        self.params = {"m": 1.0, "l": 1.0, "gravity": 9.81, "d": 0.2}
        self.state.labels = ["theta", "dtheta"]
        self.x0 = np.array([0.5, 0.0])

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        m = params["m"]
        l = params["l"]
        gravity = params["gravity"]
        d = params["d"]
        xp = array_module(x)  # numpy or jax.numpy, chosen at call time
        q, dq = x[0], x[1]
        tau = u[0]
        # (m l^2) ddq = tau - m g l sin(q) - d dq
        ddq = (tau - m * gravity * l * xp.sin(q) - d * dq) / (m * l**2)
        return xp.array([dq, ddq])

    def h(self, x, u, t=0, params=None):
        return x


pendulum = Pendulum()

# A plain NumPy call: f is just a function of (x, u).
x = np.array([0.5, 0.0])
u = np.array([0.0])
print("dx/dt =", pendulum.f(x, u))

## 3. What is JAX tracing?

JAX does not differentiate Python source text. Instead it **traces**: it runs your function once on *abstract* placeholder values ("tracers") and records the sequence of primitive operations into a small graph (a `jaxpr`). It can then transform that graph — `jit` it (compile to XLA), `grad` it, `jacfwd`/`jacrev` it for Jacobians, or `vmap` it over a batch.

Tracing only works if the function is **pure**. If `f` mutated `self`, read a clock, or branched on the *value* of a tracer, the recorded graph would be wrong or tracing would fail outright. This is exactly why the stateless contract from Section 1 matters: *"stateless `f`" is the precondition for "differentiable `f`".*

We compile the pendulum with the JAX backend. `compile(backend="jax")` traces `f`/`h` once and hands back an evaluator with JIT-compiled methods. We set `enable_x64=True` so derivatives are computed in float64.

In [ ]:
import jax
import jax.numpy as jnp

from minilink.core.backends import configure_jax

configure_jax(enable_x64=True)  # float64 for accurate derivatives

evaluator = pendulum.compile(backend="jax")  # traces f/h once, JIT-compiles to XLA

# A single operating point reused throughout the derivative sections.
x_op = jnp.array([0.5, 0.0])
u_op = jnp.array([0.0])
t_op = jnp.array(0.0)

print("dx/dt =", evaluator.f(x_op, u_op, t_op))

## 4. ∂f/∂x and ∂f/∂u — linearization (the A and B matrices)

The Jacobians of `f` with respect to the state and the input are the **A** and **B** matrices of the local linear model $\dot{\delta x} = A\,\delta x + B\,\delta u$. The eigenvalues of `A` report local stability; the pair $(A, B)$ drives controllability tests and LQR / pole-placement design (`minilink.control.lqr.lqr_gain` is built on exactly these). Both are the *same* one-liner — `jacfwd` of `f` — differing only in which argument is varied.

In [ ]:
A = np.asarray(jax.jacfwd(lambda x: evaluator.f(x, u_op, t_op))(x_op))
B = np.asarray(jax.jacfwd(lambda u: evaluator.f(x_op, u, t_op))(u_op))
print("A = df/dx =\n", A)
print("B = df/du =\n", B)
print("local eigenvalues:", np.linalg.eigvals(A))

# (A, B) -> controllability matrix [B, A B] for this 2-state system.
ctrb = np.column_stack([B, A @ B])
print("rank of [B, A B] =", np.linalg.matrix_rank(ctrb), "(2 => controllable)")

`minilink` exposes this through `analysis.linearize`, which can use either exact autodiff (`method="jax"`) or central finite differences (`method="fd"`). They agree — but the autodiff version is exact, while finite differences trade accuracy against step size. Same physics function, two ways to read its slope.

In [ ]:
from minilink.analysis.linearize import linearize_matrices

A_jax, B_jax, _, _ = linearize_matrices(pendulum, np.asarray(x_op), np.asarray(u_op), method="jax")
A_fd, B_fd, _, _ = linearize_matrices(pendulum, np.asarray(x_op), np.asarray(u_op), method="fd")

print("A (jax autodiff) =\n", A_jax)
print("A (finite diff)  =\n", A_fd)
print("max |jax - fd| =", np.max(np.abs(A_jax - A_fd)))
print("matches hand-rolled jacfwd:", np.allclose(A_jax, A) and np.allclose(B_jax, B))

## 5. ∂f/∂params — parameter sensitivity

This is the direction that hidden state would make impossible. Because the parameters are an explicit argument, we can differentiate `f` with respect to **the physics itself**. The evaluator returns `∂f/∂θ` as a pytree mirroring the `params` dict — one sensitivity vector per parameter. Below we read off how the angular acceleration `ddq` responds to each constant, and cross-check against central finite differences.

In [ ]:
params = dict(pendulum.params)
jac = evaluator.jacobian_f_params(x_op, u_op, t_op, params)

eps = 1e-6
print(f"{'param':>10} {'autodiff d(ddq)/dtheta':>24} {'finite diff':>16}")
for key in params:
    hi = dict(params); hi[key] = params[key] + eps
    lo = dict(params); lo[key] = params[key] - eps
    fd = (evaluator.f_p(x_op, u_op, t_op, hi) - evaluator.f_p(x_op, u_op, t_op, lo)) / (2 * eps)
    ad = np.asarray(jac[key])
    print(f"{key:>10} {float(ad[1]):>24.6f} {float(fd[1]):>16.6f}")

## 6. Use case: parameter identification & tuning

Sensitivity to parameters is what makes **system identification** a gradient problem. Suppose we logged states and inputs but do not know the true gravity and damping. We minimize the *equation-error* loss

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_k \big\| f(x_k, u_k, t_k;\, \theta) - \dot{x}_k \big\|^2$$

by gradient descent, with the gradient `∇_θ L` produced automatically by `jax.value_and_grad`. We use `jax.vmap` to evaluate `f` across the whole dataset at once — again only possible because `f` is pure.

Here we generate the "measured" data from the true model (in practice the derivatives come from numerically differentiating logged states), corrupt our initial guess of `gravity` and `d`, and watch them converge.

In [ ]:
# --- 1. Generate measured data from the true pendulum ----------------------
dt = 0.02
n_samples = 400
ts = dt * jnp.arange(n_samples)
us = 0.5 * jnp.sin(0.8 * ts).reshape(-1, 1)

f_jit = evaluator.get_f_jit()  # JIT-compiled f(x, u, t)


def rk4_step(x, u_k, t_k, dt):
    k1 = f_jit(x, u_k, t_k)
    k2 = f_jit(x + 0.5 * dt * k1, u_k, t_k + 0.5 * dt)
    k3 = f_jit(x + 0.5 * dt * k2, u_k, t_k + 0.5 * dt)
    k4 = f_jit(x + dt * k3, u_k, t_k + dt)
    return x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def rollout(carry, ut):
    u_k, t_k = ut
    x_next = rk4_step(carry, u_k, t_k, dt)
    return x_next, carry


x0 = jnp.array([0.5, 0.0])
_, xs = jax.lax.scan(rollout, x0, (us, ts))
dxs = jax.vmap(f_jit, in_axes=(0, 0, 0))(xs, us, ts)  # "measured" derivatives

# --- 2. Identify gravity and damping by gradient descent -------------------
f_p = evaluator.get_f_p_jit()  # JIT-compiled parametric f(x, u, t, params)


def loss(theta):
    p = {"m": 1.0, "l": 1.0, "gravity": theta[0], "d": theta[1]}
    dx_hat = jax.vmap(f_p, in_axes=(0, 0, 0, None))(xs, us, ts, p)
    return jnp.mean((dx_hat - dxs) ** 2)


loss_and_grad = jax.jit(jax.value_and_grad(loss))

theta = jnp.array([7.0, 1.0])  # wrong initial guess for (gravity, damping)
lr = 2.0
print(f"{'iter':>6} {'loss':>14} {'gravity':>10} {'damping':>10}")
for i in range(151):
    value, grad = loss_and_grad(theta)
    if i % 25 == 0:
        print(f"{i:>6} {float(value):>14.3e} {float(theta[0]):>10.4f} {float(theta[1]):>10.4f}")
    theta = theta - lr * grad

print(f"\nidentified gravity = {float(theta[0]):.4f}  (true 9.81)")
print(f"identified damping = {float(theta[1]):.4f}  (true 0.20)")

## 7. Use case: optimization & planning (differentiate through a rollout)

The same purity that lets us differentiate `f` lets us differentiate an entire **trajectory** built from `f`. An RK4 rollout is just `f` composed with itself, so the map from any rollout input to a final-state cost is differentiable end-to-end. Below are the two minimalist faces of this — same goal (put the pendulum's final angle on a target), two different decision variables:

1. **Optimize the input sequence `u`.** Hold the physics fixed and search for the torque history that lands the final angle on the target. This is the core of single-shooting trajectory optimization / MPC: the gradient is `∂(final-state cost)/∂(u-sequence)`.
2. **Optimize a natural parameter, open-loop.** Apply *no* torque (`u = 0`) and instead tune one physical constant — the rod length — so the *freely swinging* pendulum reaches the target at the final time. Here the gradient is `∂(final-state cost)/∂(params)`: design / tuning rather than control.

In both cases the gradient is produced automatically by `jax.value_and_grad`; only the decision variable changes. We reuse `rk4_step` (frozen params) and `f_p` (parametric) from Section 6.

In [ ]:
# --- Showcase 1: optimize the input sequence u to hit a target final angle ---
H1 = 40
dt1 = 0.05
x_down = jnp.array([0.0, 0.0])  # start hanging down, at rest
theta_target_1 = 2.0            # desired final angle [rad]


def final_state_u(u_seq):
    def step(x, u_scalar):
        x_next = rk4_step(x, jnp.array([u_scalar]), 0.0, dt1)
        return x_next, x_next
    x_final, traj = jax.lax.scan(step, x_down, u_seq)
    return x_final, traj


def cost_u(u_seq):
    x_final, _ = final_state_u(u_seq)
    # land the final angle on target, with a light effort penalty
    return (x_final[0] - theta_target_1) ** 2 + 1e-4 * jnp.sum(u_seq ** 2)


value_and_grad_u = jax.jit(jax.value_and_grad(cost_u))
u_seq = jnp.zeros(H1)
for _ in range(1500):
    _, g = value_and_grad_u(u_seq)
    u_seq = u_seq - 0.6 * g  # gradient descent on the whole input sequence

theta_reached_1 = float(final_state_u(u_seq)[0][0])
print(f"target  final angle: {theta_target_1:.3f} rad")
print(f"reached final angle: {theta_reached_1:.3f} rad   (by optimizing the u-sequence)")

In [ ]:
# --- Showcase 2: open-loop (u = 0), tune the rod length to hit a target ------
H2 = 30
dt2 = 0.01            # 0.3 s of free swing (under a quarter period)
theta0 = 1.0          # released from 1.0 rad, at rest
theta_target_2 = 0.7  # desired angle at the final time
u_zero = jnp.array([0.0])  # no torque: purely natural dynamics


def final_state_l(l_value):
    # only the length varies; everything else stays at its physical value
    p = {"m": 1.0, "l": l_value, "gravity": 9.81, "d": 0.2}

    def step(x, _):
        k1 = f_p(x, u_zero, 0.0, p)
        k2 = f_p(x + 0.5 * dt2 * k1, u_zero, 0.0, p)
        k3 = f_p(x + 0.5 * dt2 * k2, u_zero, 0.0, p)
        k4 = f_p(x + dt2 * k3, u_zero, 0.0, p)
        x_next = x + (dt2 / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        return x_next, x_next
    x_final, traj = jax.lax.scan(step, jnp.array([theta0, 0.0]), None, length=H2)
    return x_final, traj


def cost_l(l_value):
    x_final, _ = final_state_l(l_value)
    return (x_final[0] - theta_target_2) ** 2


value_and_grad_l = jax.jit(jax.value_and_grad(cost_l))
l_value = 1.0
for _ in range(300):
    _, g = value_and_grad_l(l_value)
    l_value = l_value - 2.0 * g  # gradient descent on the rod length

theta_reached_2 = float(final_state_l(l_value)[0][0])
print(f"target  final angle: {theta_target_2:.3f} rad")
print(f"reached final angle: {theta_reached_2:.3f} rad   (by tuning the rod length, u = 0)")
print(f"optimized length   : {float(l_value):.3f} m     (started from 1.000 m)")

In [ ]:
# Visualize both planned trajectories.
import matplotlib.pyplot as plt

_, traj1 = final_state_u(u_seq)
_, traj2 = final_state_l(l_value)
traj1 = np.asarray(traj1)
traj2 = np.asarray(traj2)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(dt1 * np.arange(H1), traj1[:, 0])
ax[0].axhline(theta_target_1, ls="--", color="k", lw=0.8, label="target")
ax[0].set_title("1) optimize u-sequence")
ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("theta [rad]"); ax[0].legend()
ax[1].plot(dt2 * np.arange(H2), traj2[:, 0])
ax[1].axhline(theta_target_2, ls="--", color="k", lw=0.8, label="target")
ax[1].set_title("2) tune length, u = 0")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("theta [rad]"); ax[1].legend()
fig.tight_layout()
plt.show()

## 8. Recap

We wrote the physics **once** as a pure function `f(x, u, t, params)` and then asked an autodiff engine for its derivative in every direction. Each direction is a different engineering tool:

| Derivative | What it is | What it powers |
|---|---|---|
| `∂f/∂x` | A matrix | local linearization, stability, LQR |
| `∂f/∂u` | B matrix | control authority, controllability, LQR/pole placement |
| `∂f/∂params` | parameter sensitivity | identification, tuning, sensitivity analysis |
| `∂(rollout)/∂u` | trajectory gradient | trajectory optimization, MPC, planning |
| `∂(rollout)/∂params` | design gradient | open-loop design, tuning constants to meet trajectory goals |

**The throughline:** a stateless, functional `f` is the precondition for tracing, tracing is the precondition for autodiff, and autodiff turns one model definition into linearization, estimation, identification, tuning, and optimization — with no separately maintained, hand-derived gradient code. Write the model; get the calculus for free.